<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">4. ETL with the DataFrame API</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 4.2 Demo - Flight Data ETL with the DataFrame API
### Duration - 29 Minutes

This demonstration walks through common ETL operations using the flights dataset created during the required setup. We cover data loading, cleaning, transformation, and analysis using the DataFrame API.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Implement common ETL operations using Spark DataFrames
- Handle data cleaning and type conversion under Spark 4 ANSI mode
- Create derived features through transformations
- Use different column reference methods
- Work with user defined functions (UDFs)

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations explore the Spark UI and cluster internals, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run (including the flights table), sets your default catalog, and sets the default schema to `intro_spark`.

In [0]:
%run ../Includes/Classroom-Setup-4

### Flight Data Processing Requirements

We are going to act on a typical ETL requirement: land a cleaned, enriched version of the raw flight data as a new table.

**Source data** - the `flights` table in your `intro_spark` schema (synthetic flight information generated during setup, with intentionally messy values)

**Target** - a table named `cleaned_and_enriched_flights` with this schema:

| Column Name | Data Type | Description |
|-------------|-----------|-------------|
| __`FlightDateTime`__ | `TIMESTAMP_NTZ` | Datetime of the flight, derived from the `Year`, `Month`, `DayofMonth`, and `DepTime` fields in the source data |
| __`FlightNum`__ | `INT` | Flight number |
| __`ElapsedTimeDiff`__ | `INT` | Difference between actual and scheduled elapsed time, derived from `ActualElapsedTime` and `CRSElapsedTime` |
| __`ArrDelayCategory`__ | `STRING` | One of `On Time`, `Slight Delay`, `Moderate Delay`, `Severe Delay`, based on the value of `ArrDelay` |

## B. Data Loading and Inspection

First, let's load and inspect the flight data.

In [0]:
%python
# Read the Flights Table
flights_df = spark.read.table("flights")

In [0]:
%python
# Print the Schema
flights_df.printSchema()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Several columns that should be numeric - <code>DepTime</code>, <code>ActualElapsedTime</code>, <code>ArrDelay</code> - arrived as strings. This is common with data exported from older systems, and cleaning it up is the core of this demo.</p>
</div>

In [0]:
%python
# Visually Inspect a Subset of the Data
display(flights_df.limit(10))

In [0]:
%python
# Select Only the Columns We Need
# Remember: filter early, filter often
flights_required_cols_df = flights_df.select(
    "Year",
    "Month",
    "DayofMonth",
    "DepTime",
    "FlightNum",
    "ActualElapsedTime",
    "CRSElapsedTime",
    "ArrDelay")

# Alternatively we could have used the drop() method to remove the columns we didn't want

In [0]:
%python
# Count the Source Records
initial_count = flights_required_cols_df.count()

print(f"Source data has {initial_count} records")

## C. Analyzing Invalid Values

The string columns we intend to perform mathematical operations on (`ArrDelay`, `ActualElapsedTime`, `DepTime`) may contain nulls or invalid values such as `NA`.

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#9888;&#65039;</span>
<div>
<strong style="color: #e65100; font-size: 1.1em">ANSI Mode and Casting</strong>
<p style="margin: 8px 0 0 0; color: #333">Spark 4 runs in ANSI SQL mode by default: <code>CAST('NA' AS INT)</code> raises an error instead of silently returning <code>NULL</code>. We therefore use <code>TRY_CAST</code>, which returns <code>NULL</code> for invalid values, letting us find and handle them deliberately.</p>
</div>
</div>
</div>

In [0]:
%python
# Register a Temp View with Cast Columns
# Register the DataFrame as a temporary view with TRY_CAST columns for analysis
flights_required_cols_df.selectExpr(
    "Year",
    "Month",
    "DayofMonth",
    "TRY_CAST(DepTime AS INT) AS DepTime",
    "FlightNum",
    "TRY_CAST(ActualElapsedTime AS INT) AS ActualElapsedTime",
    "CRSElapsedTime",
    "TRY_CAST(ArrDelay AS INT) AS ArrDelay"
).createOrReplaceTempView("flights_temp")

In [0]:
%python
# Count Null Values with Spark SQL
# Use the COUNT_IF SQL function to count invalid (now null) values per column
invalid_counts_sql = spark.sql('''
SELECT
    COUNT_IF(Year IS NULL) AS Null_Year_Count,
    COUNT_IF(Month IS NULL) AS Null_Month_Count,
    COUNT_IF(DayofMonth IS NULL) AS Null_DayOfMonth_Count,
    COUNT_IF(DepTime IS NULL) AS Null_DepTime_Count,
    COUNT_IF(FlightNum IS NULL) AS Null_FlightNum_Count,
    COUNT_IF(ActualElapsedTime IS NULL) AS Null_ActualElapsedTime_Count,
    COUNT_IF(CRSElapsedTime IS NULL) AS Null_CRSElapsedTime_Count,
    COUNT_IF(ArrDelay IS NULL) AS Null_ArrDelay_Count
FROM flights_temp
''')

display(invalid_counts_sql)

### Comparing Spark SQL to DataFrame API Operations

Spark SQL queries and their equivalent operations in the DataFrame API are evaluated to the same physical plans. Let's prove this.

In [0]:
%python
# Count Null Values with the DataFrame API
# This is the equivalent of the preceding Spark SQL query using the DataFrame API
from pyspark.sql.functions import col, sum, when

# Make sure to work with the same temporary view that the SQL is using
flights_temp_df = spark.table("flights_temp")

invalid_counts_df = flights_temp_df.select(
    sum(when(col("Year").isNull(), 1).otherwise(0)).alias("Null_Year_Count"),
    sum(when(col("Month").isNull(), 1).otherwise(0)).alias("Null_Month_Count"),
    sum(when(col("DayofMonth").isNull(), 1).otherwise(0)).alias("Null_DayOfMonth_Count"),
    sum(when(col("DepTime").isNull(), 1).otherwise(0)).alias("Null_DepTime_Count"),
    sum(when(col("FlightNum").isNull(), 1).otherwise(0)).alias("Null_FlightNum_Count"),
    sum(when(col("ActualElapsedTime").isNull(), 1).otherwise(0)).alias("Null_ActualElapsedTime_Count"),
    sum(when(col("CRSElapsedTime").isNull(), 1).otherwise(0)).alias("Null_CRSElapsedTime_Count"),
    sum(when(col("ArrDelay").isNull(), 1).otherwise(0)).alias("Null_ArrDelay_Count")
)

display(invalid_counts_df)

In [0]:
%python
# Explain the SQL Version
invalid_counts_sql.explain()

In [0]:
%python
# Explain the DataFrame Version
invalid_counts_df.explain()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Compare the two physical plans printed above - they are the same. Spark SQL and the DataFrame API are two front ends to the same Catalyst optimizer and execution engine, so you can choose whichever is clearer for the task at hand.</p>
</div>

## D. Data Cleaning

The flights data contains some invalid and missing values. Let's find them and clean them (in this case, we will drop them).

In [0]:
%python
# Drop Rows with Null CRSElapsedTime
# To drop rows where any specified columns are null, use the na.drop DataFrame method
non_null_flights_df = flights_required_cols_df.na.drop(
    how="any",
    subset=["CRSElapsedTime"]
)

In [0]:
%python
# Remove Rows with Invalid Numeric Strings
from pyspark.sql.functions import col

# Keep only rows where the string columns hold valid integers
flights_with_valid_data_df = non_null_flights_df.filter(
    col("ArrDelay").try_cast("integer").isNotNull() &
    col("ActualElapsedTime").try_cast("integer").isNotNull() &
    col("DepTime").try_cast("integer").isNotNull()
)

In [0]:
%python
# Cast the String Columns to Integers
# Now that we know the columns contain only valid integer values, cast them
# from strings to integers (replacing the existing columns)
clean_flights_df = (flights_with_valid_data_df
    .withColumn("ArrDelay", col("ArrDelay").try_cast("integer"))
    .withColumn("ActualElapsedTime", col("ActualElapsedTime").try_cast("integer")))

clean_flights_df.printSchema()

## E. Data Enrichment

Now let's create the derived columns from the requirements: a proper flight timestamp, a delay category, and the elapsed time difference.

In [0]:
%python
# Derive the FlightDateTime Column
# Derive FlightDateTime from the Year, Month, DayofMonth, and DepTime columns,
# then drop the constituent columns
from pyspark.sql.functions import col, make_timestamp_ntz, lpad, substr, lit

flights_with_datetime_df = clean_flights_df.withColumn(
    "FlightDateTime",
    make_timestamp_ntz(
        col("Year"),
        col("Month"),
        col("DayofMonth"),
        substr(lpad(col("DepTime"), 4, "0"), lit(1), lit(2)).try_cast("integer"),
        substr(lpad(col("DepTime"), 4, "0"), lit(3), lit(2)).try_cast("integer"),
        lit(0)
    )
).drop("Year", "Month", "DayofMonth", "DepTime")

display(flights_with_datetime_df.limit(10))

In [0]:
%python
# Categorize Arrival Delays
# Categorize the ArrDelay column into: On Time, Slight Delay, Moderate Delay, Severe Delay
from pyspark.sql.functions import when, col

enriched_flights_df = (
    flights_with_datetime_df
    .withColumn(
        "ArrDelayCategory",
        when(col("ArrDelay") <= 0, "On Time")
        .when(col("ArrDelay") <= 15, "Slight Delay")
        .when(col("ArrDelay") <= 60, "Moderate Delay")
        .otherwise("Severe Delay")
    )
    .drop("ArrDelay")
)

display(enriched_flights_df.limit(10))

In [0]:
%python
# Derive the ElapsedTimeDiff Column
# Derive ElapsedTimeDiff from the ActualElapsedTime and CRSElapsedTime columns
from pyspark.sql.functions import col

enriched_flights_df = enriched_flights_df.withColumn(
    "ElapsedTimeDiff", col("ActualElapsedTime") - col("CRSElapsedTime")
).drop("ActualElapsedTime", "CRSElapsedTime")

display(enriched_flights_df.select("FlightNum", "FlightDateTime", "ElapsedTimeDiff").limit(10))

### Visualize the Delay Categories

PySpark in Spark 4 includes native plotting: call `df.plot` directly on a DataFrame (backed by plotly) without converting to pandas first.

In [0]:
%python
# Plot Delay Categories with Native Plotting
# Aggregate, then plot directly from the PySpark DataFrame (new in Spark 4)
delay_counts_df = enriched_flights_df.groupBy("ArrDelayCategory").count()

delay_counts_df.plot.bar(x="ArrDelayCategory", y="count")

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Other Visualization Options</strong>
<p style="margin: 8px 0 0 0; color: #333">You can also use the <strong>+</strong> -> <strong>Visualization</strong> option in any <code>display()</code> result, or ask the Databricks Assistant to generate chart code for you.</p>
</div>
</div>
</div>

## F. Analyzing Delays with Different Column References

Let's analyze our delay categories while exercising the column referencing approaches from the lecture.

In [0]:
%python
# Reference Columns Directly
# Direct string references
display(enriched_flights_df.select("FlightNum", "ArrDelayCategory").limit(100))

In [0]:
%python
# Reference Columns with Column Objects
from pyspark.sql.functions import col

# Column objects support methods like alias()
display(
    enriched_flights_df
    .select(col("FlightNum").alias("flight_number"), col("ArrDelayCategory"))
    .limit(100)
)

In [0]:
%python
# Reference Columns with String Expressions
# selectExpr accepts SQL expressions
display(
    enriched_flights_df
    .selectExpr("FlightNum", "ElapsedTimeDiff", "ElapsedTimeDiff > 0 AS LongerThanScheduled")
    .limit(100)
)

## G. Working with UDFs

Let's use a vectorized pandas UDF to calculate the z-score (standard deviations from the mean) of the elapsed time difference for each flight.

In [0]:
%python
# Apply a Vectorized Pandas UDF
from pyspark.sql.functions import pandas_udf

# Pandas UDF (vectorized) - operates on batches of rows as pandas Series
@pandas_udf("double")
def normalized_diff(diff_series):
    return (diff_series - diff_series.mean()) / diff_series.std()

udf_example_df = enriched_flights_df.withColumn(
    "diff_normalized", normalized_diff("ElapsedTimeDiff")
)

display(udf_example_df.limit(100))

# Note: in practice, prefer built-in functions over UDFs when possible

## H. Putting It All Together

Let's combine everything into a single chained operation that reads from the source, cleans, enriches, and saves to a new target table (overwriting any existing data).

In [0]:
-- Drop the Target Table if it Exists
DROP TABLE IF EXISTS cleaned_and_enriched_flights;

In [0]:
%python
# Run the Full ETL Pipeline
from pyspark.sql.functions import col, make_timestamp_ntz, lpad, substr, lit, when, pandas_udf

@pandas_udf("double")
def normalized_diff(diff_series):
    return (diff_series - diff_series.mean()) / diff_series.std()

(spark.read.table("flights")
    # Project and convert the columns we need
    .selectExpr(
        "Year",
        "Month",
        "DayofMonth",
        "TRY_CAST(DepTime AS INT) AS DepTime",
        "FlightNum",
        "TRY_CAST(ActualElapsedTime AS INT) AS ActualElapsedTime",
        "CRSElapsedTime",
        "TRY_CAST(ArrDelay AS INT) AS ArrDelay"
    )
    # Drop any rows with invalid or missing values
    .na.drop()
    # Derive the flight timestamp, then drop its constituent columns
    .withColumn(
        "FlightDateTime",
        make_timestamp_ntz(
            col("Year"),
            col("Month"),
            col("DayofMonth"),
            substr(lpad(col("DepTime"), 4, "0"), lit(1), lit(2)).try_cast("integer"),
            substr(lpad(col("DepTime"), 4, "0"), lit(3), lit(2)).try_cast("integer"),
            lit(0)
        )
    )
    .drop("Year", "Month", "DayofMonth", "DepTime")
    # Derive the elapsed time difference
    .withColumn("ElapsedTimeDiff", col("ActualElapsedTime") - col("CRSElapsedTime"))
    .drop("ActualElapsedTime", "CRSElapsedTime")
    # Categorize the arrival delay
    .withColumn(
        "ArrDelayCategory",
        when(col("ArrDelay") <= 0, "On Time")
        .when(col("ArrDelay") <= 15, "Slight Delay")
        .when(col("ArrDelay") <= 60, "Moderate Delay")
        .otherwise("Severe Delay")
    )
    .drop("ArrDelay")
    # Add the normalized difference using our pandas UDF
    .withColumn("diff_normalized", normalized_diff("ElapsedTimeDiff"))
    # Write to the target table
    .write
    .mode("overwrite")
    .saveAsTable("cleaned_and_enriched_flights"))

In [0]:
-- Query the Target Table
SELECT * FROM cleaned_and_enriched_flights LIMIT 10;

## Key Takeaways

**Data cleaning best practices:**

- Validate and clean data types early, using `try_cast()` under Spark 4 ANSI mode
- Handle missing values appropriately
- Document cleaning assumptions

**Data enrichment:**

- Create meaningful derived columns driven by business requirements
- Use built-in functions where possible, and user defined functions where necessary

**Pipeline composition:**

- Because transformations are lazy and return new DataFrames, an entire pipeline can be expressed as one readable chain ending in a single write action

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>